# Referral prediction — train & export

Predictive-only pipeline: **P(any donation referral)** (classifier, ROC-AUC CV) and **predicted referral count** (regressor on `log1p`, MAE on log CV). Stage-A features = CSV columns minus a fixed leakage denylist (`feature_config.py`). Stage-B = columns with positive permutation importance on the best regressor; both final pipelines refit on that subset.

**Run this notebook** with the working directory set to the `referral_prediction` folder (or use `%cd` below).

In [ ]:
# Set notebook cwd to this folder if imports fail.
import pandas as pd
from feature_config import stage_a_columns
from train_models import train_and_export, DEFAULT_DATA
import json

In [ ]:
df0 = pd.read_csv(DEFAULT_DATA)
print("Rows:", len(df0))
stage_a = stage_a_columns(df0.columns)
print("Stage-A feature count:", len(stage_a))
print("Features:", stage_a)

In [ ]:
# rolling_months=None uses full history; e.g. 24 for last 24 months from max(created_at)
results = train_and_export(rolling_months=None)
print(json.dumps(results["meta"], indent=2))

In [ ]:
pd.DataFrame(results["classifier_cv"]).T

In [ ]:
pd.DataFrame(results["regressor_cv"]).T

## After training

- Artifacts: `artifacts/referral_classifier.joblib`, `referral_count_regressor.joblib`, `selected_feature_names.json`
- Batch score: `python predict_referrals.py --csv ... --out scored.csv`
- Weekly schedule helper: `python build_candidates.py --days 7`